# OFAT hyperparameter sensitivity — ModernTCN on realized volatilityAnchors every hyperparameter at the tuned configuration and sweeps **one at a time** over the gridOptuna searched. Each point is trained over 5 seeds, so every curve carries a mean and a spreadrather than a single draw.### Local, not globalAn OFAT curve says how the loss moves when you step away from the tuned point in **one** direction.It cannot see interactions — a pair of hyperparameters that only matters jointly is invisible here.The curves are valid *around* the optimum, which is what makes them the right companion to theOptuna study rather than a replacement for it: the study has the whole space, this has theneighbourhood in readable form.### What it costsThe full sweep is about **40 configurations × 5 seeds = 200 training runs** — a couple of hours ona Colab GPU, longer on CPU. Three things keep that manageable:* Results append to a CSV **on Drive after every point**, so a dropped runtime costs the point in  flight and nothing else. Re-running this notebook skips what is already recorded.* `PARAMS` below selects which factors to sweep; start with a few if you want a first look.* Section 4 prints the plan and the exact commands before anything trains.Unlike the Optuna notebook, nothing here needs SQLite, so the results file lives on Drive directly —plain CSV appends are safe on the FUSE mount.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

DRIVE_DIR = "/content/drive/MyDrive/ModernTCN_RV"    # same folder the search notebook uses
os.makedirs(DRIVE_DIR, exist_ok=True)
print("results will be saved to:", DRIVE_DIR)

## 2. Repository and packages

In [ ]:
import subprocess, sys

REPO   = "https://github.com/Mr0022/ModernTCNt.git"
BRANCH = "claude/moderntcn-aggregation-log-ptj8ao"
ROOT   = "/content/ModernTCNt"

if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, ROOT], check=True)
else:
    subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], check=False)

# os.chdir, not %cd: it moves the Python process, so the ! cells below inherit it.
os.chdir(os.path.join(ROOT, "ModernTCN-Long-term-forecasting"))
print("working directory:", os.getcwd())

import torch
print("torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

## 3. Configuration**The anchor.** Taken from `results_optuna/optuna_h1_best.json` in your Drive folder — what thesearch notebook writes — so a finished Optuna run feeds this sweep with nothing copied by hand. Ifthat file is not there, the sweep falls back to the tuned defaults baked into `run.py`, and says sowhen it starts.

In [ ]:
HORIZON  = 1           # h: the sweep pivots on this horizon's tuned configuration
ITR      = 5           # seeds per point
EPOCHS   = 50          # cap per run; early stopping usually ends one sooner
PATIENCE = 10

# Which factors to sweep. Comment out any you want to skip -- the plan and the
# figures both follow this list.
PARAMS = [
    "seq_len", "patch_size", "patch_stride", "dim", "ffn_ratio",
    "large_size", "small_size", "num_blocks",
    "dropout", "head_dropout", "learning_rate", "batch_size", "revin",
]

# Written straight to Drive: ordinary CSV appends, one per completed point.
RESULTS = os.path.join(DRIVE_DIR, f"ofat_results_h{HORIZON}.csv")
FIGDIR  = os.path.join(DRIVE_DIR, f"ofat_figures_h{HORIZON}")
ANCHOR  = os.path.join(DRIVE_DIR, "results_optuna", f"optuna_h{HORIZON}_best.json")
os.makedirs(FIGDIR, exist_ok=True)

print(f"results : {RESULTS}")
print(f"figures : {FIGDIR}")
print(f"anchor  : {ANCHOR}")
print("          " + ("found — sweeping around the tuned configuration"
                      if os.path.exists(ANCHOR) else
                      "not found — will fall back to run.py's tuned defaults"))

## 4. The planPrints every configuration that will be trained, and the exact `run.py` command for each, withouttraining anything. Points already in the results CSV are left out, so this doubles as a progresscheck on a partly finished sweep.

In [ ]:
common = (
    f"--pred_len {HORIZON} --itr {ITR} --train_epochs {EPOCHS} --patience {PATIENCE} "
    f"--out {RESULTS} --anchor_json {ANCHOR} --num_workers 2"
)

!python sensitivity/ofat_sensitivity.py {common} --params {" ".join(PARAMS)} --dry_run

## 5. Run the sweepOne `ofat_sensitivity.py` call per factor, so progress is legible and an interruption is cheap.Each completed point appends to the CSV on Drive immediately.Interrupting this cell is safe. Re-run it and the sweep picks up where it stopped.

In [ ]:
import time

started = time.time()
for i, p in enumerate(PARAMS, 1):
    print("\n" + "#" * 78)
    print(f"#  [{i}/{len(PARAMS)}]  sweeping {p}"
          f"   ({(time.time()-started)/60:.0f} min elapsed)")
    print("#" * 78, flush=True)
    !python sensitivity/ofat_sensitivity.py {common} --params {p}

print(f"\nSweep finished in {(time.time()-started)/60:.0f} min.")
print(f"Results: {RESULTS}")

## 6. FiguresFour figures, written to Drive as both PDF (for the thesis) and PNG (for reading here):| figure | what it shows ||---|---|| `ofat_response_mse` | one panel per factor: the 5-seed mean, a ±1 sd band, the individual seeds, and the tuned anchor || `ofat_response_qlike` | the same on QLIKE, the finance loss || `ofat_tornado_mse` | how far each factor moves the loss either side of the anchor, ranked || `ofat_sensitivity_bar` | each factor's range as a percentage of the anchor — which knobs matter at all |A **hollow marker** means `patch_stride` had to be clamped to `patch_size` there, so that pointdiffers from the anchor in two factors rather than one and the OFAT reading does not strictly hold.

In [ ]:
!python sensitivity/ofat_plots.py --results {RESULTS} --outdir {FIGDIR} \
        --anchor_json {ANCHOR} --horizon {HORIZON} --metric mse

In [ ]:
from IPython.display import Image, display
import glob

for f in sorted(glob.glob(os.path.join(FIGDIR, "*.png"))):
    print("\n" + os.path.basename(f))
    display(Image(filename=f, width=1000))

## 7. Ranking tableThe same information as the sensitivity bar, as numbers to quote. `range_%` is how much the metricmoves across a factor's whole grid, relative to the anchor; `best_delta` is the improvementavailable by moving that one factor alone — if it is near zero everywhere, the search found agenuine local optimum.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(RESULTS, dtype={"value": str})
anchor_rows = df[df.param == "anchor"]

rows = []
for metric in ("mse", "qlike"):
    a = float(anchor_rows[metric].mean())
    for p in [x for x in PARAMS if not df[df.param == x].empty]:
        g = df[df.param == p].groupby("value")[metric].mean()
        means = np.append(g.values, a)          # the anchor is that curve's centre point
        rows.append({
            "metric": metric, "param": p,
            "anchor": a, "min": means.min(), "max": means.max(),
            "range_%": (means.max() - means.min()) / a * 100,
            "best_delta": a - means.min(),
        })

rank = pd.DataFrame(rows).sort_values(["metric", "range_%"], ascending=[True, False])
display(rank.set_index(["metric", "param"]).round(6))
rank.to_csv(os.path.join(FIGDIR, "ofat_ranking.csv"), index=False)
print("\nSaved:", os.path.join(FIGDIR, "ofat_ranking.csv"))

## 8. What is on Drive| path | what ||---|---|| `ofat_results_h{h}.csv` | every seed of every point — the raw sweep, resumable || `ofat_figures_h{h}/*.pdf` | the four figures, vector, for the thesis || `ofat_figures_h{h}/ofat_summary.csv` | per-point mean and sd, tidy || `ofat_figures_h{h}/ofat_ranking.csv` | the ranking table above |To extend the sweep later — more factors, or the same factors at another horizon — change `PARAMS`or `HORIZON` and re-run. Each horizon keeps its own results file and figure folder, and finishedpoints are never retrained.